# 07 — End-to-End Smoke Test & Submission Checklist

Run this notebook last. It verifies the final serialized model, predictions, unseen-category handling, web artifacts, and required project files.

In [1]:
from pathlib import Path
import sys, json, os

def _find_root():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "data" / "bonbanh_usedcar_dataset.csv").exists() and (p / "src").exists():
            return p
    raise FileNotFoundError("Project root not found. Keep notebooks inside the CarValue_AI_Complete_Project folder.")

PROJECT_ROOT = _find_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /mnt/data/CarValue_AI_Complete_Project_v2


In [2]:
import json, py_compile
import numpy as np
import pandas as pd
from src.carvalue_utils import ensure_modeling_artifacts, get_train_test, load_bundle, TARGET, FEATURES, compatibility_subset, compatible_values, make_user_input, market_reference, apply_market_guardrail

model_df, reference_year = ensure_modeling_artifacts(PROJECT_ROOT)
train, test = get_train_test(model_df)
model = load_bundle(PROJECT_ROOT / "models" / "final_model.joblib")


In [3]:
# Predict a deterministic sample from the held-out grouped test set
sample = test.sample(n=min(20, len(test)), random_state=42).copy()
pred = model.predict(sample)
assert len(pred) == len(sample)
assert np.isfinite(pred).all()
assert (pred >= 0).all()

smoke = sample[["brand","model",TARGET]].copy()
smoke["prediction_vnd"] = pred
smoke["absolute_error_vnd"] = np.abs(smoke[TARGET] - smoke["prediction_vnd"])
smoke.to_csv(PROJECT_ROOT / "artifacts" / "smoke_test_predictions.csv", index=False)
display(smoke.head(10))
print("PASS: final model returns finite non-negative prices.")


,brand,model,price_vnd,prediction_vnd,absolute_error_vnd
16263,Kia,Cerato 2.0 AT Premium,4.200000e+08,4.177751e+08,2.224896e+06
14204,Mitsubishi,Attrage 1.2 MT Eco,1.750000e+08,1.765949e+08,1.594924e+06
8068,Mercedes-Benz,GLC 300 4Matic,2.529000e+09,2.540615e+09,1.161536e+07
13507,Toyota,Raize G 1.0 CVT,4.650000e+08,4.363447e+08,2.865531e+07
17736,Hyundai,Custin Cao Cấp 2.0T,7.790000e+08,7.808266e+08,1.826600e+06
115,Toyota,Fortuner 2.7V 4X2 AT,4.200000e+08,4.244018e+08,4.401781e+06
23161,Hyundai,SantaFe 2.2L,6.690000e+08,6.868761e+08,1.787608e+07
8779,Porsche,911 Carrera 4 GTS,9.979000e+09,9.516730e+09,4.622702e+08
3434,Lexus,LX 570,2.179000e+09,2.182946e+09,3.945905e+06
11569,Hyundai,SantaFe Premium 2.4L HTRAC,6.950000e+08,7.158610e+08,2.086099e+07


PASS: final model returns finite non-negative prices.


In [4]:
# Unseen category handling test
unseen = sample.iloc[[0]][FEATURES].copy()
unseen["brand"] = "__UNSEEN_BRAND_FOR_TEST__"
unseen["model"] = "__UNSEEN_MODEL_FOR_TEST__"
unseen_pred = model.predict(unseen)
assert np.isfinite(unseen_pred).all()
print("PASS: unseen category test prediction =", f"{float(unseen_pred[0]):,.0f} VND")


PASS: unseen category test prediction = 544,542,037 VND


In [5]:
# Syntax-check app without needing to launch Streamlit
py_compile.compile(str(PROJECT_ROOT / "app.py"), doraise=True)
print("PASS: app.py syntax is valid.")

required = [
    PROJECT_ROOT / "data" / "bonbanh_usedcar_dataset.csv",
    PROJECT_ROOT / "artifacts" / "CarValue_AI_model_candidate.csv",
    PROJECT_ROOT / "artifacts" / "metrics_summary.csv",
    PROJECT_ROOT / "artifacts" / "web_options.json",
    PROJECT_ROOT / "models" / "final_model.joblib",
    PROJECT_ROOT / "models" / "final_model_metadata.json",
    PROJECT_ROOT / "app.py",
    PROJECT_ROOT / "requirements.txt",
    PROJECT_ROOT / "README.md",
]
missing = [str(p) for p in required if not p.exists()]
assert not missing, f"Missing required deliverables: {missing}"
print("PASS: all core deliverables exist.")


PASS: app.py syntax is valid.
PASS: all core deliverables exist.


In [6]:
# Web-specific compatibility and plausibility smoke test
vf3 = compatibility_subset(model_df, "VinFast", "VF3 Plus", 2025)
if len(vf3):
    assert compatible_values(vf3, "body_type") == ["Hatchback"]
    assert compatible_values(vf3, "fuel_type") == ["Điện"]
    assert [int(x) for x in compatible_values(vf3, "seats")] == [4]
    assert compatible_values(vf3, "engine_size_l") == []
    vf3_input = make_user_input(reference_year, "VinFast", "VF3 Plus", 2025, 15000, "Lắp ráp trong nước", "Hatchback", "Số tự động", "Điện", np.nan, "Trắng", 4, "RFD - Dẫn động cầu sau")
    vf3_raw = float(model.predict(vf3_input)[0])
    vf3_ref = market_reference(model_df, "VinFast", "VF3 Plus", 2025)
    vf3_final, _ = apply_market_guardrail(vf3_raw, vf3_ref)
    assert vf3_ref["q10"] * 0.75 <= vf3_final <= vf3_ref["q90"] * 1.25
    print("PASS: VF3 compatibility + plausibility test:", f"{vf3_final/1e6:.1f}M VND")

PASS: VF3 compatibility + plausibility test: 242.9M VND


## Final checklist

- [x] EDA and data-quality audit
- [x] Repaired modeling dataset
- [x] Leakage-safe grouped split
- [x] Median + Ridge baselines
- [x] Random Forest + XGBoost + CatBoost
- [x] MAE / RMSE / R² / RMSLE
- [x] Error by price tier and major brand
- [x] Feature importance
- [x] Frozen final model
- [x] Streamlit demo app
- [x] Smoke test and serialized artifacts

**Project boundary:** the target is the online asking price, not the negotiated transaction price.